# Python Memory Model: References, Mutability, and Aliasing

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Understanding Python's execution model requires deconstructing the relationship between **names**, **references**, and **heap objects**.

- In Python, variables are not memory locations that hold values; they are symbolic pointers bound to objects in dynamic heap memory.
- Assignment (`y = x`) never creates an object copy; it binds an additional name to the exact same memory address.
- Mutating a shared mutable object through one name immediately alters the value observed by all other aliases.

### Learning Objectives
1. Differentiate between object identity (`is`) and value equality (`==`).
2. Trace object memory addresses with `id()` and `hex(id())`.
3. Master the copy protocol: reference assignment vs. shallow copy (`copy.copy`) vs. deep copy (`copy.deepcopy`).
4. Understand Python's parameter passing mechanism: **Call by Object Reference**.
5. Avoid the critical anti-pattern of mutating a list while iterating over it.
6. Understand CPython runtime optimizations: Small Integer Caching ($[-5, 256]$) and variable shadowing.

In [ ]:
# Identity vs. Equality
a: list[int] = [1, 2, 3]
b: list[int] = a  # Aliasing: shared pointer
c: list[int] = [1, 2, 3]  # Distinct object with equivalent value

print(f"a address: {hex(id(a))}")
print(f"b address: {hex(id(b))}")
print(f"c address: {hex(id(c))}")

assert a is b  # Identical memory address
assert a is not c  # Different memory allocations
assert a == c  # Equivalent contents

# Mutating through alias b
b.append(4)
assert a == [1, 2, 3, 4]  # Mutation visible through name a!
print("Aliasing assertions passed successfully.")

## Shallow Copy vs. Deep Copy

When working with compound or nested data structures (e.g., lists of lists, matrices, model weights):
- **Shallow Copy (`copy.copy(x)` or `x[:]`):** Constructs a new container object, but populates it with *references* to the child objects found in the original.
- **Deep Copy (`copy.deepcopy(x)`):** Recursively constructs a new container and duplicates all nested objects at every level of the hierarchy.

In [ ]:
import copy

original_matrix: list[list[int]] = [[1, 2], [3, 4]]

# Create shallow and deep copies
shallow = copy.copy(original_matrix)
deep = copy.deepcopy(original_matrix)

# Container addresses are distinct in both
assert id(original_matrix) != id(shallow)
assert id(original_matrix) != id(deep)

# Inner child element addresses
assert id(original_matrix[0]) == id(shallow[0])  # Shallow copy shares nested objects!
assert id(original_matrix[0]) != id(deep[0])  # Deep copy creates isolated child objects!

# Mutating an inner element
original_matrix[0][0] = 999
assert shallow[0][0] == 999  # Leaks into shallow copy!
assert deep[0][0] == 1  # Deep copy remains completely isolated!
print("Copy protocol assertions validated successfully.")

## Function Argument Passing: Call by Object Reference

Python uses **Call by Object Reference** (or Call by Assignment):
- If an argument is mutable (such as a `list`) and a function invokes an in-place mutating method (e.g., `.append()`, `[i] = v`), the caller's object is modified.
- If the parameter name is reassigned (`lst = lst + [v]`), the local name is bound to a new object, severing the link to the caller's object.

In [ ]:
def modify_in_place(target_list: list[int], item: int) -> None:
    target_list.append(item)


def reassign_locally(target_list: list[int], item: int) -> list[int]:
    target_list = [*target_list, item]  # Creates a new local list
    return target_list


my_data = [1, 2, 3]
modify_in_place(my_data, 4)
assert my_data == [1, 2, 3, 4]  # Modified in place

returned_data = reassign_locally(my_data, 5)
assert my_data == [1, 2, 3, 4]  # Caller's data unchanged!
assert returned_data == [1, 2, 3, 4, 5]
print("Argument passing semantics validated.")

## The Mutation-During-Iteration Hazard and CPython Optimizations

### 1. Mutating While Iterating
Modifying a list while iterating over it with a `for` loop produces internal index displacement, leading to skipped items or infinite loops. To mutate safely, always iterate over a shallow copy (`for item in data[:]:`) or construct a new list via comprehension.

### 2. Small Integer Caching
CPython pre-allocates an array of singleton integer objects for all values in the range $[-5, 256]$. Any variable assigned an integer in this range points to the exact same pre-allocated memory address.

In [ ]:
# Mutation during iteration hazard
items: list[int] = [1, 2, 3, 4, 5]
# Safe idiom: iterate over shallow slice copy
for x in items[:]:
    if x == 3:
        items.append(300)
assert 300 in items

# Small integer caching verification
int_a: int = 250
int_b: int = 250
assert int_a is int_b  # Within [-5, 256] range: singleton cache hit

large_a: int = 1000
large_b: int = 1000
assert large_a == large_b
# Outside [-5, 256], distinct objects may be allocated
print("Small integer caching and safe iteration validated.")

## Practical Lab Exercise: Immutable Data Pipeline Architecture

Implement a defensive data transformation pipeline `safe_pipeline(records: list[dict], transform_fn)` that guarantees the original input records are not mutated under any circumstances.

In [ ]:
from collections.abc import Callable
from typing import Any


def safe_pipeline(
    records: list[dict[str, Any]],
    transform_fn: Callable[[dict[str, Any]], dict[str, Any]],
) -> list[dict[str, Any]]:
    # Deep copy input records to enforce strict boundary immutability
    cloned_records = copy.deepcopy(records)
    transformed = []
    for record in cloned_records:
        result = transform_fn(record)
        transformed.append(result)
    return transformed


# Test pipeline isolation
raw_samples = [
    {"id": 1, "features": [0.1, 0.2]},
    {"id": 2, "features": [0.8, 0.9]},
]


def aggressive_transform(item: dict[str, Any]) -> dict[str, Any]:
    item["features"].append(99.9)
    item["status"] = "processed"
    return item


output_samples = safe_pipeline(raw_samples, aggressive_transform)

# Original samples remain untouched
assert len(raw_samples[0]["features"]) == 2
assert "status" not in raw_samples[0]
# Output contains modifications
assert len(output_samples[0]["features"]) == 3
assert output_samples[0]["status"] == "processed"
print("Pipeline isolation lab exercise verified successfully!")